## Máster en Big Data y Data Science

### Metodologías de gestión y diseño de proyectos de big data

#### Sesión Práctica 01 - Ejemplo mlflow

---

En esta libreta se realizan una serie de pasos básicos para verificar la operatoria a seguir con respecto al registro del entrenamiento y evaluación de modelos de aprendizaje automático utilizando la librería mlflow.

JF: Esta celda la he añadido yo, para poder volver a reejecutar las celdas siguientes por segunda vez. Lo que hace es borrar los experimentos para no tener que cambiar los nombres cada vez. Otra opción es cambiar los nombres en los experimentos de las celdas siguientes para que se generen nuevos.

Celda 1: conexión y comprobación--

In [1]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

import os
import mlflow
from mlflow import MlflowClient
from mlflow.entities import ViewType

client = MlflowClient()
print("Tracking URI:", mlflow.get_tracking_uri())
print("Usuario configurado:", bool(os.getenv("MLFLOW_TRACKING_USERNAME")))
print("Token configurado:", bool(os.getenv("MLFLOW_TRACKING_PASSWORD")))

Tracking URI: https://dagshub.com/sortaria24/13MBID_Proyectos.mlflow
Usuario configurado: True
Token configurado: True


Celda 2: estado actual (incluidos los borrados)

In [2]:
for e in client.search_experiments(view_type=ViewType.ALL):
    runs = client.search_runs([e.experiment_id], run_view_type=ViewType.ALL, max_results=1000)
    print(e.experiment_id, e.name, e.lifecycle_stage, f"-> {len(runs)} runs")

3 Clustering-Ejemplos_2 active -> 1 runs
2 Clustering-Ejemplos_ deleted -> 1 runs
1 Clustering-Ejemplos deleted -> 1 runs
0 Default active -> 4 runs


Celda 3: borrar todos los runs activos de todos los experimentos

In [3]:
def borrar_runs(exp_id):
    n, token = 0, None
    while True:
        page = client.search_runs([exp_id], run_view_type=ViewType.ACTIVE_ONLY,
                                  max_results=500, page_token=token)
        for r in page:
            client.delete_run(r.info.run_id)
            n += 1
        token = page.token
        if not token:
            break
    return n

for e in client.search_experiments(view_type=ViewType.ACTIVE_ONLY):
    print(e.experiment_id, e.name, "-> runs borrados:", borrar_runs(e.experiment_id))

3 Clustering-Ejemplos_2 -> runs borrados: 1
0 Default -> runs borrados: 4


Celda 4: borrar todos los experimentos activos excepto el 0

In [4]:
for e in client.search_experiments(view_type=ViewType.ACTIVE_ONLY):
    if e.experiment_id == "0":
        print("Se mantiene el experimento 0 (Default)")
        continue
    try:
        client.delete_experiment(e.experiment_id)
        print("Borrado:", e.experiment_id, e.name)
    except Exception as err:
        print("No se pudo borrar", e.experiment_id, e.name, "->", err)

Borrado: 3 Clustering-Ejemplos_2
Se mantiene el experimento 0 (Default)


Celda 5: comprobación final

In [5]:
for e in client.search_experiments(view_type=ViewType.ALL):
    runs_activos = client.search_runs([e.experiment_id], run_view_type=ViewType.ACTIVE_ONLY, max_results=1000)
    print(e.experiment_id, e.name, e.lifecycle_stage, f"-> {len(runs_activos)} runs activos")

3 Clustering-Ejemplos_2 deleted -> 0 runs activos
2 Clustering-Ejemplos_ deleted -> 0 runs activos
1 Clustering-Ejemplos deleted -> 0 runs activos
0 Default active -> 0 runs activos


===

In [6]:
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

import mlflow
import numpy as np

from sklearn.datasets import load_iris
from sklearn.cluster import DBSCAN, KMeans
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
)

Se realiza en primera instancia la lectura del conjunto de datos que será uno genérico.

In [7]:
db = load_iris()
features = db.data
target = db.target

Se ejecuta un método de clusterización haciendo uso de "autolog" de parte de la librería. Esta es una opción básica de registro que es de utilidad para los métodos de sklearn.

In [8]:
# Opción básica
mlflow.autolog()

# Clustering con DBSCAN.
dbscan = DBSCAN(eps=0.5, min_samples=5)

dbscan.fit_predict(features)

2026/10/11 08:34:40 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.
2026/10/11 08:34:40 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '3179ac8f8b07410aa43b541c3a3b7587', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2026/10/11 08:34:40 WARNING mlflow.sklearn: Failed to infer model signature: the trained model does not have a `predict` or `transform` function, which is required in order to infer the signature
2026/10/11 08:34:42 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/10/11 08:34:42 WARNING mlflow.sklearn: Mod

🏃 View run colorful-rat-819 at: https://dagshub.com/sortaria24/13MBID_Proyectos.mlflow/#/experiments/0/runs/3179ac8f8b07410aa43b541c3a3b7587
🧪 View experiment at: https://dagshub.com/sortaria24/13MBID_Proyectos.mlflow/#/experiments/0


array([ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
        0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
        0,  0,  0,  0,  0,  0,  0, -1,  0,  0,  0,  0,  0,  0,  0,  0,  1,
        1,  1,  1,  1,  1,  1, -1,  1,  1, -1,  1,  1,  1,  1,  1,  1,  1,
       -1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,
        1,  1, -1,  1,  1,  1,  1,  1, -1,  1,  1,  1,  1, -1,  1,  1,  1,
        1,  1,  1, -1, -1,  1, -1, -1,  1,  1,  1,  1,  1,  1,  1, -1, -1,
        1,  1,  1, -1,  1,  1,  1,  1,  1,  1,  1,  1, -1,  1,  1, -1, -1,
        1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1])

In [9]:
# Clustering con Kmeans.
kmeans = KMeans(n_clusters=3, random_state=42)

kmeans.fit_predict(features)

# davies_bouldin_score(features, kmeans.fit_predict(features))

2026/10/11 08:35:13 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID 'd2afbd5ee21a4a228051847c4ccf537e', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2026/10/11 08:35:14 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/10/11 08:35:16 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/10/11 08:35:18 WARNING mlflow.sklearn: Training metrics will not be recorded because training labels were not specified. To automatically record training me

🏃 View run wise-swan-512 at: https://dagshub.com/sortaria24/13MBID_Proyectos.mlflow/#/experiments/0/runs/d2afbd5ee21a4a228051847c4ccf537e
🧪 View experiment at: https://dagshub.com/sortaria24/13MBID_Proyectos.mlflow/#/experiments/0


array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 0, 2, 0, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 0, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 0, 2, 0, 0, 0, 0, 2, 0, 0, 0,
       0, 0, 0, 2, 2, 0, 0, 0, 0, 2, 0, 2, 0, 2, 0, 0, 2, 2, 0, 0, 0, 0,
       0, 2, 0, 0, 0, 0, 2, 0, 0, 0, 2, 0, 0, 0, 2, 0, 0, 2], dtype=int32)

Se muestran ejemplos con una denominación diferente para la experimentación a realizar a fin de dejar mayor registro.

In [11]:
# Se puede hacer una experimentación nominal

exp_name = 'Clustering-Ejemplos_3'
exp_id = mlflow.set_experiment(exp_name).experiment_id

with mlflow.start_run(experiment_id=exp_id, run_name="Kmeans - K=2"):
    modelo_clusters = KMeans(n_clusters=2)
    trained_model = modelo_clusters.fit(features)
    cluster_labels = trained_model.labels_
    score = silhouette_score(features, cluster_labels)
    #save parameter
    mlflow.log_param('value_of_k', 2)
    #save metric
    mlflow.log_metric('silhoutte_score', score)
    #save model
    input_example = np.array([features[0]])  # Example input
    mlflow.sklearn.log_model(trained_model, "Clustering_Model", input_example=input_example)
    #end current run
    mlflow.end_run()

2026/10/11 08:36:00 INFO mlflow.tracking.fluent: Experiment with name 'Clustering-Ejemplos_3' does not exist. Creating a new experiment.
2026/10/11 08:36:06 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/10/11 08:36:09 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/10/11 08:36:11 WARNING mlflow.sklearn: Training metrics will not be recorded because training labels were not specified. To automatically record training metrics, provide training labels as inputs to the model training function.
2026/10/11 08:36:12 WARNING mlflow.models.model: `a

🏃 View run Kmeans - K=2 at: https://dagshub.com/sortaria24/13MBID_Proyectos.mlflow/#/experiments/4/runs/57b6732cdc434cdaa05f3ff0d20458d7
🧪 View experiment at: https://dagshub.com/sortaria24/13MBID_Proyectos.mlflow/#/experiments/4


In [12]:
with mlflow.start_run(experiment_id=exp_id, run_name="Kmeans - K=3"):
    modelo_clusters = KMeans(n_clusters=3)
    trained_model = modelo_clusters.fit(features)
    cluster_labels = trained_model.labels_
    score = silhouette_score(features, cluster_labels)
    score_2 = davies_bouldin_score(features, cluster_labels)
    #save parameter
    mlflow.log_param('value_of_k', 3)
    #save metric
    mlflow.log_metric('silhoutte_score', score)
    mlflow.log_metric('davies_bouldin_score', score_2)
    #save model
    input_example = np.array([features[0]])  # Example input
    mlflow.sklearn.log_model(trained_model, "Clustering_Model", input_example=input_example)
    #end current run
    mlflow.end_run()

2026/10/11 08:36:43 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/10/11 08:36:45 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/10/11 08:37:00 WARNING mlflow.sklearn: Training metrics will not be recorded because training labels were not specified. To automatically record training metrics, provide training labels as inputs to the model training function.
2026/10/11 08:37:08 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/10/11 08:37:21 WARNING mlflow.utils.environment: Failed to resolve installed

🏃 View run Kmeans - K=3 at: https://dagshub.com/sortaria24/13MBID_Proyectos.mlflow/#/experiments/4/runs/5952bb98718f490fa5b9efa5f8d6bfcf
🧪 View experiment at: https://dagshub.com/sortaria24/13MBID_Proyectos.mlflow/#/experiments/4


In [13]:
with mlflow.start_run(experiment_id=exp_id, run_name="DBSCAN"):
    modelo_clusters = DBSCAN(eps=0.5, min_samples=5)
    trained_model = modelo_clusters.fit(features)
    cluster_labels = trained_model.labels_
    score=silhouette_score(features, cluster_labels)
    #save parameter
    mlflow.log_param('min_samples', 5)
    mlflow.log_param('eps', 0.5)
    #save metric
    mlflow.log_metric('silhoutte_score', score)
    #save model
    input_example = np.array([features[0]])  # Example input
    mlflow.sklearn.log_model(trained_model, "Clustering_Model", input_example=input_example)
    #end current run
    mlflow.end_run()

2026/10/11 08:50:09 WARNING mlflow.sklearn: Failed to infer model signature: the trained model does not have a `predict` or `transform` function, which is required in order to infer the signature
2026/10/11 08:50:24 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/10/11 08:50:24 WARNING mlflow.sklearn: Model was missing function: predict. Not logging python_function flavor!
2026/10/11 08:50:27 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/10/11 08:50:30 WARNING mlflow.sklearn: Training metrics will not be recorded because training labels wer

🏃 View run DBSCAN at: https://dagshub.com/sortaria24/13MBID_Proyectos.mlflow/#/experiments/4/runs/87609de6bc0a434e843db400c4b29589
🧪 View experiment at: https://dagshub.com/sortaria24/13MBID_Proyectos.mlflow/#/experiments/4


Este comando es el que permite visualizar la interfaz de mlflow en un navegador web. Se debe considerar ejecutarlo en la misma ubicación que contiene al directorio `mlruns`.

In [ ]:
!mlflow ui